#**1. Import library**

Menyiapkan library yang diperlukan, menghubungkan Google Drive, kemudian membaca dataset perpustakaan yang masih dalam kondisi mentah untuk digunakan dalam proses data cleansing.

In [78]:
# untuk mengolah data tabel
import pandas as pd
import numpy as np
# regex untuk mencari, mencocokkan, atau mengganti teks berdasarkan pola.
import re
from google.colab import drive

drive.mount('/content/drive')

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

file_path = '/content/drive/MyDrive/Colab Notebooks/DatMin/data_perpustakaan_kotor.xlsx'
df = pd.read_excel(file_path)
df_asli = df.copy()   # simpan salinan mentah untuk perbandingan

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# **2. Pemeriksaan Awal Dataset**
Memeriksa jumlah baris dan kolom serta menampilkan beberapa data awal untuk mengetahui kondisi dataset sebelum dilakukan proses cleansing, seperti adanya penulisan yang tidak seragam dan data kosong.

In [79]:
print("Jumlah baris dan kolom:", df.shape)
display(df.head(10))

Jumlah baris dan kolom: (44, 11)


,id_peminjaman,id_nama,nama_peminjam,judul_buku,kategori_buku,tanggal_pinjam,tanggal_kembali,status_pengembalian,denda_peminjam,kondisi_buku,petugas_perpus
0,PJM2031,NAM523,kartika wijaya,SEJARAH NUSANTARA,Nonfiksi,20-01-2024,NaN,Belum Kembali,0,Baik,bambang t.
1,PJM2003,NAM553,Fajar Wijaya,DATA MINING: KONSEP DAN APLIKASI,referensi,2025-01-16,NaN,Belum Kembali,0,NaN,SITI N.
2,PJM2037,NAM533,"Fajar Kurniawan, S.Kom.",Kamus Besar Bahasa Indonesia,Referensi,2024-09-13,14/09/2024,Sudah Kembali,0,Rusak Ringan,Bambang T
3,PJM2035,NAM517,Dewi Hidayat,laskar pelangi,FIKSI,07/20/2025,09/08/2025,Terlambat,2000,Rusak Ringan,RINA W.
4,PJM2008,NAM544,Bapak Oki Kurniawan,Sej. Nusantara,NON FIKSI,2024-12-21,02/01/2025,TERLAMBAT,20000,Baik,Agus Setiawan
5,PJM2034,NAM506,Ibu Indah Saputra,SEJARAH NUSANTARA,NaN,2025-03-11,2025-03-12,Terlambat,26000,Rusak Ringan,RINA W.
6,PJM2015,NAM508,Putri Hidayat,jurnal ilmu komputer vol.12,Jurnal,2024-11-15,19-11-2024,sudah kembali,0,hilang,NaN
7,PJM2009,NAM550,Joko Pratama,sejarah nusantara,non-fiksi,02/11/2025,14/02/2025,TERLAMBAT,16000,rusak ringan,agus s.
8,PJM2023,NAM556,Gita Lestari,Laskar Pelangi,FIKSI,05-08-2024,NaN,Belum Kembali,0,Rusak Berat,DEWI P.
9,PJM2024,NAM535,Nanda,Laskar Pelangi,FIKSI,04/14/2025,NaN,Belum Kembali,0,BAIK,Ibu Siti


# **3. Normalisasi dan Standarisasi Data Teks**

Menyeragamkan penulisan pada data teks agar memiliki format yang konsisten. Proses ini meliputi penghapusan spasi berlebih, penyamaan huruf, penghapusan gelar pada nama, serta penyamaan penulisan nama petugas dan judul buku yang memiliki variasi berbeda.

In [80]:
kolom_teks = ["nama_peminjam", "judul_buku", "kategori_buku",
              "status_pengembalian", "kondisi_buku", "petugas_perpus"]

for kolom in kolom_teks:
    df[kolom] = (df[kolom].astype("string")
                 .str.strip()
                 .str.replace(r"\s+", " ", regex=True))

# --- nama peminjam: hapus sapaan & gelar (depan/belakang) ---
gelar_depan = r"^(Bapak|Ibu|Bu|Pak|Dr|Drs|Dra|Ir|Prof)\.?\s+"
gelar_belakang = r",?\s+(S\.?\s?Kom|S\.?\s?E|S\.?\s?T|S\.?\s?Pd|S\.?\s?Si|S\.?\s?H|S\.?\s?Sos|M\.?\s?Kom|M\.?\s?Pd|M\.?\s?M|M\.?\s?T)\.?$"
df["nama_peminjam"] = (df["nama_peminjam"]
    .str.replace(gelar_depan, "", regex=True, case=False)
    .str.replace(gelar_belakang, "", regex=True, case=False)
    .str.title())
# --- petugas: hapus sapaan, ambil nama depan ---
df["petugas_perpus"] = (df["petugas_perpus"]
    .str.replace(r"^(Bapak|Ibu|Bu|Pak)\.?\s+", "", regex=True, case=False)
    .str.split().str[0].str.title())
# --- judul: samakan alias ke satu nama baku ---
def baku_judul(x):
    if pd.isna(x): return x
    k = re.sub(r"[^a-z0-9]", "", x.lower())          # buang spasi & tanda baca
    k = k.replace("dan", "").replace("&", "")
    peta = {
        "jtivol3": "Jurnal Teknik Informatika Vol. 3",
        "jurnalteknikinformatikavol3": "Jurnal Teknik Informatika Vol. 3",
        "jikvol12": "Jurnal Ilmu Komputer Vol. 12",
        "jurnalilmukomputervol12": "Jurnal Ilmu Komputer Vol. 12",
        "kbbi": "Kamus Besar Bahasa Indonesia",
        "kamusbesarbahasaindonesia": "Kamus Besar Bahasa Indonesia",
        "negeri5menara": "Negeri Lima Menara",
        "negerilimamenara": "Negeri Lima Menara",
        "sejnusantara": "Sejarah Nusantara",
        "sejarahnusantara": "Sejarah Nusantara",
        "dataminingkonsepaplikasi": "Data Mining: Konsep dan Aplikasi",
    }
    return peta.get(k, x.title())
df["judul_buku"] = df["judul_buku"].apply(baku_judul)
# --- kategori ---
df["kategori_buku"] = (df["kategori_buku"].str.lower()
    .replace({"non fiksi": "nonfiksi", "non-fiksi": "nonfiksi", "ref.": "referensi", "ref": "referensi"})
    .str.title())
df["status_pengembalian"] = df["status_pengembalian"].str.title()
df["kondisi_buku"] = df["kondisi_buku"].str.title()

for k in ["judul_buku", "kategori_buku", "status_pengembalian", "kondisi_buku", "petugas_perpus"]:
    print(f"{k}: {sorted(df[k].dropna().unique())}")

judul_buku: ['Atomic Habits', 'Bumi Manusia', 'Data Mining: Konsep dan Aplikasi', 'Filosofi Kopi', 'Jurnal Ilmu Komputer Vol. 12', 'Jurnal Teknik Informatika Vol. 3', 'Kamus Besar Bahasa Indonesia', 'Laskar Pelangi', 'Negeri Lima Menara', 'Pengantar Basis Data', 'Sapiens', 'Sejarah Nusantara']
kategori_buku: ['Fiksi', 'Jurnal', 'Nonfiksi', 'Referensi']
status_pengembalian: ['Belum Kembali', 'Sudah Kembali', 'Terlambat']
kondisi_buku: ['Baik', 'Hilang', 'Rusak Berat', 'Rusak Ringan']
petugas_perpus: ['Agus', 'Bambang', 'Dewi', 'Rina', 'Siti']


# **4. Standarisasi Format Tanggal**

Menyeragamkan berbagai format tanggal menjadi format tanggal yang konsisten serta memastikan tanggal pengembalian tidak lebih awal dari tanggal peminjaman.

In [81]:
# Ubah kolom menjadi tanggal saja
df["tanggal_pinjam"] = pd.to_datetime(
    df["tanggal_pinjam"],
    errors="coerce"
).dt.strftime("%Y-%m-%d")

df["tanggal_kembali"] = pd.to_datetime(
    df["tanggal_kembali"],
    errors="coerce"
).dt.strftime("%Y-%m-%d")

# Cek hasil
print(df[["tanggal_pinjam", "tanggal_kembali"]])


   tanggal_pinjam tanggal_kembali
0      2024-01-20             NaN
1             NaN             NaN
2             NaN      2024-09-14
3             NaN      2025-08-09
4             NaN      2025-01-02
5             NaN             NaN
6             NaN             NaN
7             NaN      2025-02-14
8      2024-08-05             NaN
9             NaN             NaN
10            NaN             NaN
11            NaN             NaN
12            NaN             NaN
13            NaN             NaN
14            NaN             NaN
15            NaN             NaN
16            NaN             NaN
17            NaN             NaN
18            NaN             NaN
19            NaN      2025-07-02
20            NaN      2025-07-02
21     2024-11-15             NaN
22            NaN             NaN
23     2025-04-04             NaN
24            NaN             NaN
25            NaN             NaN
26            NaN             NaN
27            NaN      2025-08-07
28            

/tmp/ipykernel_1804/1258330845.py:2: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df["tanggal_pinjam"] = pd.to_datetime(
/tmp/ipykernel_1804/1258330845.py:7: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df["tanggal_kembali"] = pd.to_datetime(


# **5. Cleansing Data Denda Peminjaman**
Memastikan nilai denda peminjaman berupa angka yang valid. Nilai yang bukan angka diubah menjadi kosong, sedangkan denda bernilai negatif diubah menjadi 0.

In [82]:
print("Sebelum:"); display(df[["denda_peminjam"]].describe().T)
# errors="coerce": nilai yang bukan angka diubah jadi NaN, bukan error
df["denda_peminjam"] = pd.to_numeric(df["denda_peminjam"], errors="coerce")
print("Denda negatif:", (df["denda_peminjam"] < 0).sum())
df.loc[df["denda_peminjam"] < 0, "denda_peminjam"] = 0  # denda negatif -> 0
df["denda_peminjam"] = df["denda_peminjam"].fillna(0).astype(int) # kosong -> 0, jadi integer

print("Sesudah:"); display(df[["denda_peminjam"]].describe().T)

Sebelum:


,count,mean,std,min,25%,50%,75%,max
denda_peminjam,44.0,5477.272727,9670.11372,-5000.0,0.0,0.0,8000.0,30000.0


Denda negatif: 1
Sesudah:


,count,mean,std,min,25%,50%,75%,max
denda_peminjam,44.0,5590.909091,9573.019698,0.0,0.0,0.0,8000.0,30000.0


# **6. Menghapus Data Duplikat**

Mengidentifikasi data dengan ID peminjaman yang sama dan menghapus data duplikat setelah proses normalisasi, sehingga setiap peminjaman hanya memiliki satu data yang tersimpan.

In [83]:
print("ID kosong   :", df["id_peminjaman"].isna().sum())
print("ID duplikat :", df["id_peminjaman"].duplicated().sum())

# Menampilkan data dengan ID yang duplikat
duplikat = df[
    df["id_peminjaman"].duplicated(keep=False)
].sort_values("id_peminjaman")

display(
    duplikat[
        [
            "id_peminjaman",
            "id_nama",
            "nama_peminjam",
            "judul_buku",
            "tanggal_pinjam",
            "tanggal_kembali",
            "status_pengembalian",
            "denda_peminjam",
            "kondisi_buku",
            "petugas_perpus"
        ]
    ]
)
# Menghapus baris yang benar-benar identik
df = df.drop_duplicates().reset_index(drop=True)
# Menangani konflik nama pada PJM2024
df.loc[
    (df["id_peminjaman"] == "PJM2024") &
    (df["nama_peminjam"] == "Nanda"),
    "nama_peminjam"
] = "Nanda Pratama"
# Menghapus duplikat PJM2024 setelah normalisasi nama
df = df.drop_duplicates().reset_index(drop=True)
# Validasi hasil cleansing
print(
    "ID duplikat setelah cleansing :",
    df["id_peminjaman"].duplicated().sum()
)
print("Jumlah baris setelah cleansing :", len(df))
# Menampilkan ID yang masih memiliki duplikat
duplikat_id = df[
    df["id_peminjaman"].duplicated(keep=False)
].sort_values("id_peminjaman")

if not duplikat_id.empty:
    print("\nID yang masih memiliki data berbeda:")

    display(
        duplikat_id[
            [
                "id_peminjaman",
                "id_nama",
                "nama_peminjam",
                "judul_buku",
                "tanggal_pinjam",
                "tanggal_kembali",
                "status_pengembalian",
                "denda_peminjam",
                "kondisi_buku",
                "petugas_perpus"
            ]
        ]
    )
else:
    print("\nTidak ada ID duplikat yang tersisa.")

ID kosong   : 0
ID duplikat : 4


,id_peminjaman,id_nama,nama_peminjam,judul_buku,tanggal_pinjam,tanggal_kembali,status_pengembalian,denda_peminjam,kondisi_buku,petugas_perpus
1,PJM2003,NAM553,Fajar Wijaya,Data Mining: Konsep dan Aplikasi,NaN,NaN,Belum Kembali,0,<NA>,Siti
17,PJM2003,NAM553,Fajar Wijaya,Data Mining: Konsep dan Aplikasi,NaN,NaN,Belum Kembali,0,<NA>,Siti
19,PJM2020,NAM559,Lukman Wijaya,Jurnal Teknik Informatika Vol. 3,NaN,2025-07-02,Terlambat,26000,Baik,Bambang
20,PJM2020,NAM559,Lukman Wijaya,Jurnal Teknik Informatika Vol. 3,NaN,2025-07-02,Terlambat,26000,Baik,Bambang
10,PJM2022,NAM553,Toni Hidayat,Jurnal Teknik Informatika Vol. 3,NaN,NaN,Sudah Kembali,0,Baik,Bambang
14,PJM2022,NAM553,Toni Hidayat,Jurnal Teknik Informatika Vol. 3,NaN,NaN,Sudah Kembali,0,Baik,Bambang
9,PJM2024,NAM535,Nanda,Laskar Pelangi,NaN,NaN,Belum Kembali,0,Baik,Siti
34,PJM2024,NAM535,Nanda Pratama,Laskar Pelangi,NaN,NaN,Belum Kembali,0,Baik,Siti


ID duplikat setelah cleansing : 0
Jumlah baris setelah cleansing : 40

Tidak ada ID duplikat yang tersisa.


# **7. Penanganan Missing Value**

Menangani data yang kosong berdasarkan konteks masing-masing kolom. Data yang masih dapat ditentukan berdasarkan informasi lain akan diisi, sedangkan data yang memang belum tersedia tetap dibiarkan kosong. Kondisi buku yang tidak dapat diketahui diberi keterangan “Tidak Diketahui”.

In [84]:
# 1. Menyimpan jumlah missing value sebelum penanganan
sebelum = df.isnull().sum()
# 2. Mengisi kategori_buku berdasarkan judul_buku
peta_kat = (
    df.dropna(subset=["kategori_buku"])
      .groupby("judul_buku")["kategori_buku"]
      .agg(lambda s: s.mode()[0] if not s.mode().empty else np.nan)
)
df["kategori_buku"] = (
    df["kategori_buku"]
    .fillna(df["judul_buku"].map(peta_kat))
)
# 3. Mengisi status_pengembalian berdasarkan data pendukung
tanpa_status = df["status_pengembalian"].isna()
# Tidak ada tanggal kembali
# -> Belum Kembali
df.loc[
    tanpa_status
    & df["tanggal_kembali"].isna(),
    "status_pengembalian"
] = "Belum Kembali"
# Ada tanggal kembali dan terdapat denda
# -> Terlambat
df.loc[
    tanpa_status
    & df["tanggal_kembali"].notna()
    & (df["denda_peminjam"] > 0),
    "status_pengembalian"
] = "Terlambat"
# Ada tanggal kembali dan tidak terdapat denda
# -> Sudah Kembali
df.loc[
    tanpa_status
    & df["tanggal_kembali"].notna()
    & (df["denda_peminjam"] == 0),
    "status_pengembalian"
] = "Sudah Kembali"
# 4. Menangani nama_peminjam yang kosong
df["nama_peminjam"] = (
    df["nama_peminjam"]
    .fillna("Tidak Diketahui")
)
# 5. Menangani kondisi_buku
# Jika buku belum kembali, kondisi buku belum dapat diketahui
df.loc[
    df["status_pengembalian"] == "Belum Kembali",
    "kondisi_buku"
] = "Tidak Diketahui"

# Jika kondisi buku masih kosong, isi dengan Tidak Diketahui
df["kondisi_buku"] = (
    df["kondisi_buku"]
    .fillna("Tidak Diketahui")
)
# 6. Menangani petugas_perpus yang kosong
df["petugas_perpus"] = (
    df["petugas_perpus"]
    .fillna("Tidak Diketahui")
)
# 7. Menampilkan perbandingan missing value
sesudah = df.isnull().sum()

display(
    pd.DataFrame({
        "Sebelum": sebelum,
        "Sesudah": sesudah
    })
)
# 8. Menampilkan data yang memiliki kondisi buku
print("\nData dengan kondisi buku Tidak Diketahui:")

display(
    df[
        df["kondisi_buku"] == "Tidak Diketahui"
    ][
        [
            "id_peminjaman",
            "nama_peminjam",
            "judul_buku",
            "kategori_buku",
            "status_pengembalian",
            "kondisi_buku"
        ]
    ]
)
# 9. Menampilkan distribusi kondisi buku
print("\nDistribusi kondisi buku setelah cleansing:")

display(
    df["kondisi_buku"].value_counts()
)

,Sebelum,Sesudah
id_peminjaman,0,0
id_nama,0,0
nama_peminjam,4,0
judul_buku,0,0
kategori_buku,4,0
tanggal_pinjam,33,33
tanggal_kembali,31,31
status_pengembalian,4,0
denda_peminjam,0,0
kondisi_buku,4,0



Data dengan kondisi buku Tidak Diketahui:


,id_peminjaman,nama_peminjam,judul_buku,kategori_buku,status_pengembalian,kondisi_buku
0,PJM2031,Kartika Wijaya,Sejarah Nusantara,Nonfiksi,Belum Kembali,Tidak Diketahui
1,PJM2003,Fajar Wijaya,Data Mining: Konsep dan Aplikasi,Referensi,Belum Kembali,Tidak Diketahui
8,PJM2023,Gita Lestari,Laskar Pelangi,Fiksi,Belum Kembali,Tidak Diketahui
9,PJM2024,Nanda Pratama,Laskar Pelangi,Fiksi,Belum Kembali,Tidak Diketahui
15,PJM2006,Fajar Santoso,Filosofi Kopi,Fiksi,Belum Kembali,Tidak Diketahui
18,PJM2036,Indah Saputra,Filosofi Kopi,Fiksi,Belum Kembali,Tidak Diketahui
20,PJM2005,Joko Gunawan,Jurnal Ilmu Komputer Vol. 12,Jurnal,Belum Kembali,Tidak Diketahui
21,PJM2010,Nanda Gunawan,Atomic Habits,Nonfiksi,Sudah Kembali,Tidak Diketahui
23,PJM2030,Tidak Diketahui,Jurnal Teknik Informatika Vol. 3,Jurnal,Belum Kembali,Tidak Diketahui
26,PJM2029,Gita Kurniawan,Bumi Manusia,Fiksi,Belum Kembali,Tidak Diketahui



Distribusi kondisi buku setelah cleansing:


,count
kondisi_buku,
Tidak Diketahui,17
Baik,10
Rusak Ringan,7
Hilang,3
Rusak Berat,3


# **8. Validasi akhir**
Memeriksa kembali konsistensi data setelah proses cleansing, seperti jumlah baris, ID duplikat, nilai denda negatif, urutan tanggal, status pengembalian, kondisi buku, dan kesesuaian nama peminjam.

In [85]:
# Validasi jumlah baris
print("Baris awal -> akhir:", len(df_asli), "->", len(df))
# Validasi ID duplikat
print("ID duplikat :", df["id_peminjaman"].duplicated().sum())
# Validasi denda negatif
print("Denda negatif :", (df["denda_peminjam"] < 0).sum())
# Validasi tanggal
print(
    "kembali < pinjam :",
    (df["tanggal_kembali"] < df["tanggal_pinjam"]).sum()
)
# Validasi status dan tanggal kembali
print(
    "Status 'Belum Kembali' tapi ada tgl kembali :",
    (
        (df["status_pengembalian"] == "Belum Kembali")
        & df["tanggal_kembali"].notna()
    ).sum()
)
print(
    "Status sudah/terlambat tapi tgl kembali kosong:",
    (
        (df["status_pengembalian"] != "Belum Kembali")
        & df["tanggal_kembali"].isna()
    ).sum()
)
# Validasi kondisi buku
print(
    "Kondisi buku masih kosong:",
    df["kondisi_buku"].isna().sum()
)
print(
    "Kondisi buku 'Tidak Diketahui':",
    (df["kondisi_buku"] == "Tidak Diketahui").sum()
)
# Menampilkan ID nama yang memiliki lebih dari satu nama berbeda
print("\nid_nama dengan >1 nama berbeda:")
nama_per_id = df.groupby("id_nama")["nama_peminjam"].nunique()
anomali_id_nama = nama_per_id[nama_per_id > 1]
print(anomali_id_nama)

# Menampilkan tipe data
print("\nTipe data:")
display(df.dtypes)

Baris awal -> akhir: 44 -> 40
ID duplikat : 0
Denda negatif : 0
kembali < pinjam : 0
Status 'Belum Kembali' tapi ada tgl kembali : 0
Status sudah/terlambat tapi tgl kembali kosong: 16
Kondisi buku masih kosong: 0
Kondisi buku 'Tidak Diketahui': 17

id_nama dengan >1 nama berbeda:
id_nama
NAM507    2
NAM513    2
NAM533    2
NAM534    3
NAM545    2
NAM553    4
Name: nama_peminjam, dtype: int64

Tipe data:


,0
id_peminjaman,object
id_nama,object
nama_peminjam,string[python]
judul_buku,object
kategori_buku,string[python]
tanggal_pinjam,object
tanggal_kembali,object
status_pengembalian,string[python]
denda_peminjam,int64
kondisi_buku,string[python]


# **9. Data Enrichment**
Menambahkan informasi turunan dari data yang telah dibersihkan, yaitu lama peminjaman berdasarkan selisih tanggal pinjam dan tanggal kembali serta status denda berdasarkan nilai denda peminjaman.

In [87]:
# Menambahkan lama peminjaman
df["lama_pinjam_hari"] = (
    df["tanggal_kembali"] - df["tanggal_pinjam"]
)
# Menambahkan indikator denda
df["ada_denda"] = np.where(
    df["denda_peminjam"] > 0,
    "Ya",
    "Tidak"
)
# Cek sisa nilai kosong
print("Sisa NaN per kolom:")
print(df.isnull().sum()[lambda s: s > 0])
# Menampilkan hasil enrichment
display(
    df[
        [
            "id_peminjaman",
            "kategori_buku",
            "kondisi_buku",
            "petugas_perpus",
            "tanggal_pinjam",
            "tanggal_kembali",
            "lama_pinjam_hari",
            "ada_denda"
        ]
    ].head(10)
)

Sisa NaN per kolom:
tanggal_pinjam      33
tanggal_kembali     31
lama_pinjam_hari    40
dtype: int64


,id_peminjaman,kategori_buku,kondisi_buku,petugas_perpus,tanggal_pinjam,tanggal_kembali,lama_pinjam_hari,ada_denda
0,PJM2031,Nonfiksi,Tidak Diketahui,Bambang,2024-01-20,NaN,NaN,Tidak
1,PJM2003,Referensi,Tidak Diketahui,Siti,NaN,NaN,NaN,Tidak
2,PJM2037,Referensi,Rusak Ringan,Bambang,NaN,2024-09-14,NaN,Tidak
3,PJM2035,Fiksi,Rusak Ringan,Rina,NaN,2025-08-09,NaN,Ya
4,PJM2008,Nonfiksi,Baik,Agus,NaN,2025-01-02,NaN,Ya
5,PJM2034,Nonfiksi,Rusak Ringan,Rina,NaN,NaN,NaN,Ya
6,PJM2015,Jurnal,Hilang,Tidak Diketahui,NaN,NaN,NaN,Tidak
7,PJM2009,Nonfiksi,Rusak Ringan,Agus,NaN,2025-02-14,NaN,Ya
8,PJM2023,Fiksi,Tidak Diketahui,Dewi,2024-08-05,NaN,NaN,Tidak
9,PJM2024,Fiksi,Tidak Diketahui,Siti,NaN,NaN,NaN,Tidak


# **10. Menyimpan Data Bersih**
Menyimpan dataset yang telah melalui seluruh proses cleansing ke dalam file Excel agar dapat digunakan untuk tahap analisis atau pengolahan data selanjutnya.

In [ ]:
output_path = "/content/drive/MyDrive/Colab Notebooks/DatMin/data_perpustakaan_bersih.xlsx"

df_out = df.copy()
for k in ["tanggal_pinjam", "tanggal_kembali"]:
    df_out[k] = df_out[k].dt.date
df_out.to_excel(output_path, index=False)

print("File tersimpan di Drive :", output_path)
display(df_out.head(10))